<a href="https://colab.research.google.com/github/lyla-YijiaWu/CVResearchIntern/blob/main/Homework2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Homework 2: Sequence Chunking with BiLSTM (CoNLL-2000)**

**Course:** CSC 247/447 Natural Language Processing  
**Task:** Base NP Chunking (BIO tagging)  
**Dataset:** `conll2000` (HuggingFace `datasets`)  

> This notebook is the *student version* with TODOs. Fill in all TODO blocks.

You will implement a PyTorch pipeline end-to-end:
1) data loading + sanity checks  
2) batching with padding + masking  
3) a BiLSTM tagger  
4) evaluation with `seqeval`  
5) controlled experiments (pretrained embeddings + casing ablation)

**Deliverables**
- `Homework2.ipynb` (completed)
- `report.pdf` (short report answering the questions embedded in the notebook)

> Important: Your evaluation must ignore padded positions. Your loss must ignore padded labels via `ignore_index=-100`.


## **Part 0 — Environment**

If you use Colab, run the install cell once. If you run locally, skip it and ensure the same packages exist.

**Important Notes**

*   We use `datasets==2.19.1` because the CoNLL-2000 dataset is implemented as a dataset script. Newer versions of datasets no longer support loading dataset scripts.
*   Google Colab includes many pre-installed packages (e.g., JAX, OpenCV, SHAP). When installing pinned versions for this assignment, `pip` may print
dependency warnings about unrelated packages (e.g., NumPy/SciPy/FSSPEC).
These warnings do not affect the assignment as long as the installation
completes and `load\_dataset("conll2000")` runs successfully. **If you see such warnings, simply restart the runtime after installation.**
*   No HuggingFace authentication token is required since CoNLL-2000 is a public dataset.
*   If running locally, create a clean virtual environment before installing dependencies.




In [78]:
# Install
!pip install -U pip

In [79]:
!pip -q install datasets==2.19.1 seqeval==1.2.2 gensim==4.3.3

In [80]:
import os, random, math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from seqeval.metrics import f1_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

We **strongly encourage** running this notebook on **Google Colab** for:  
  **Free GPU access** (L4, T4, or A100 for faster training).  
    - Training **one epoch on CPU** takes **approximately 5 minutes**.  
    - Training **one epoch on GPU (L4)** takes **only a few seconds**.  

  **Pre-installed packages**, reducing setup time.

In [81]:
# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## **Part 1 — Load the Dataset (CoNLL-2000 Chunking)**

In this assignment, we use the **CoNLL-2000** chunking dataset (`conll2000`) for sequence tagging.

Each example contains:
- `tokens`: list of tokens
- `pos_tags`: POS tag IDs (optional feature)
- `chunk_tags`: chunk tag IDs (BIO scheme)

The mapping `id → string` is stored in:

- `dataset["train"].features["chunk_tags"].feature.names`


In [82]:
# TODO 1.1: Load dataset splits
dataset = load_dataset("conll2000")

# conll2000 only has train/test, so we create a validation split from train
split = dataset["train"].train_test_split(test_size=0.1, seed=42)

train_ds = split["train"]
val_ds   = split["test"]
test_ds  = dataset["test"]

print(train_ds, val_ds, test_ds)

Dataset({
    features: ['id', 'tokens', 'pos_tags', 'chunk_tags'],
    num_rows: 8043
}) Dataset({
    features: ['id', 'tokens', 'pos_tags', 'chunk_tags'],
    num_rows: 894
}) Dataset({
    features: ['id', 'tokens', 'pos_tags', 'chunk_tags'],
    num_rows: 2013
})


In [83]:
# TODO 1.2: Inspect dataset fields
ex = train_ds[0]
print("Example keys:", ex.keys())
print("First 20 tokens:", ex["tokens"][:20])

tag_names = train_ds.features["chunk_tags"].feature.names  # TODO
print("Num chunk tags:", len(tag_names))
print("First 15 chunk tags:", tag_names[:15])

Example keys: dict_keys(['id', 'tokens', 'pos_tags', 'chunk_tags'])
First 20 tokens: ['Which', 'types', 'of', 'stocks', 'are', 'most', 'likely', 'to', 'qualify', '?']
Num chunk tags: 23
First 15 chunk tags: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP']


In [84]:
# TODO 1.3: Sanity check token/tag alignment
import random
for _ in range(5):
    i = random.randint(0, len(train_ds) - 1)
    ex = train_ds[i]
    ok = len(ex["tokens"]) == len(ex["chunk_tags"])  # TODO
    print(i, len(ex["tokens"]), len(ex["chunk_tags"]), "OK" if ok else "MISMATCH")

5238 7 7 OK
912 28 28 OK
204 35 35 OK
6074 19 19 OK
2253 10 10 OK


## **Part 2 — Dataset Analysis (Q1)**

Compute and report:
- #sentences in train/validation/test
- average sentence length
- chunk tag frequency distribution


In [85]:
# TODO 2.1: Sentence counts
print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:",len(test_ds))

Train: 8043
Val: 894
Test: 2013


In [86]:
# TODO 2.2: Average sentence length (tokens)
def avg_len(ds):
    return sum(len(ex["tokens"]) for ex in ds) /len(ds)

print("Avg len train:", avg_len(train_ds))
print("Avg len val:", avg_len(val_ds))
print("Avg len test:", avg_len(test_ds))

Avg len train: 23.667536988685814
Avg len val: 23.90268456375839
Avg len test: 23.53551912568306


In [87]:
# TODO 2.3: Chunk tag frequency
from collections import Counter
tag_counter = Counter()
for ex in train_ds:
    tag_counter.update(ex["chunk_tags"])

top = tag_counter.most_common(10)
for tid, cnt in top:
    print(tid, tag_names[tid], cnt)

12 I-NP 57030
11 B-NP 49544
0 O 25030
21 B-VP 19295
13 B-PP 19106
22 I-VP 10778
3 B-ADVP 3803
17 B-SBAR 1969
1 B-ADJP 1848
2 I-ADJP 588


## **Part 3 — Preprocessing(Vocabulary + Features)**

We build:
- `word2idx` with special tokens `<PAD>=0`, `<UNK>=1`
- `tag2idx`, `idx2tag` from `chunk_tags`


In [88]:
# TODO 3.1: Build word2idx from training tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for ex in train_ds:
    for w in ex["tokens"]:
        if w not in word2idx:
            word2idx[w] = len(word2idx)
idx2word = dict((i, w) for w, i in word2idx.items())
print("Vocab size:", len(word2idx))

Vocab size: 18126


In [89]:
# TODO 3.2: Build tag mappings from dataset feature names
tag2idx = dict(zip(tag_names, range(len(tag_names))))
idx2tag = dict(zip(range(len(tag_names)), tag_names))

print("Num tags:", len(tag2idx))
print("Example tag2idx items:", list(tag2idx.items())[:5])

Num tags: 23
Example tag2idx items: [('O', 0), ('B-ADJP', 1), ('I-ADJP', 2), ('B-ADVP', 3), ('I-ADVP', 4)]


## **Part 4 — Dataset Class + Dynamic Batching**

We need a `Dataset` that converts raw tokens into tensors, and a custom `collate_fn` to pad variable-length sequences.

We pad:
- `input_ids` with 0 (`<PAD>`)
- `labels` with -100 (ignored by loss)
- `attention_mask` with 1 for real tokens, 0 for padding


### **Task 4.1: Implement `ChunkingDataset.__getitem__`**
Return:
- `input_ids` (LongTensor)
- `labels` (LongTensor)

In [90]:
class ChunkingDataset(Dataset):
    def __init__(self, hf_ds, word2idx):
        self.ds = hf_ds
        self.word2idx = word2idx

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        ex = self.ds[i]
        tokens = ex["tokens"]
        tags = ex["chunk_tags"]

        # TODO 4.1: map tokens -> ids (UNK for OOV)
        input_ids = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in tokens]

        labels = tags
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

### **Task 4.2: Implement `collate_fn`**
Pad a batch to the max sequence length and return a dict with:
- `input_ids` (B, T)
- `labels` (B, T)
- `attention_mask` (B, T)

In [91]:
def collate_fn(batch):
    T = max(x["input_ids"].shape[0] for x in batch)
    B = len(batch)

    # Complete the tensors according to the instruction
    input_ids = torch.zeros(B,T, dtype=torch.long)
    labels = torch.full((B,T), -100, dtype=torch.long)
    attn = torch.zeros(B,T, dtype=torch.long)

    for i, x in enumerate(batch):
        n = x["input_ids"].shape[0]
        input_ids[i, :n] = x["input_ids"]
        labels[i, :n] = x["labels"]
        attn[i, :n] = 1

    return {"input_ids": input_ids, "labels": labels, "attention_mask": attn}

In [92]:
train_loader = DataLoader(ChunkingDataset(train_ds, word2idx), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(ChunkingDataset(val_ds, word2idx), batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(ChunkingDataset(test_ds, word2idx), batch_size=32, shuffle=False, collate_fn=collate_fn)

# Inspect a batch
batch = next(iter(train_loader))
print(batch["input_ids"].shape, batch["labels"].shape, batch["attention_mask"].shape)

torch.Size([32, 39]) torch.Size([32, 39]) torch.Size([32, 39])


## **Part 5 — BiLSTM Tagger (Q2)**

Implement a sequence tagger:
- word embedding
- BiLSTM
- linear layer to tag logits


### **Task 5.1: Implement `forward()`**
Your forward should output logits with shape `(B, T, num_tags)`.

In [93]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, num_tags, word_dim=100, hidden_dim=128,
                 dropout=0.1, pad_idx=0):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, word_dim, padding_idx=pad_idx)

        rnn_in = word_dim
        self.dropout = nn.Dropout(dropout)

        self.rnn = nn.LSTM(
            input_size=rnn_in,hidden_size=hidden_dim,num_layers=1,
            batch_first=True, bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim *2, num_tags)

    def forward(self, input_ids):
        # TODO 5.1: forward pass
        w = self.word_emb(input_ids)
        x = w
        x = self.dropout(x)
        out, _ = self.rnn(x)
        out = self.dropout(out)
        logits = self.fc(out)
        return logits

In [94]:
# TODO 5.2: Instantiate the model (random initialization)
model = BiLSTMTagger(len(word2idx), len(tag2idx)).to(device)  # TODO
print(model)

loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)

batch = next(iter(train_loader))
logits = model(batch["input_ids"].to(device))
print("logits:", logits.shape)

BiLSTMTagger(
  (word_emb): Embedding(18126, 100, padding_idx=0)
  (dropout): Dropout(p=0.1, inplace=False)
  (rnn): LSTM(100, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=23, bias=True)
)
logits: torch.Size([32, 49, 23])


## **Part 6 — Train / Evaluate (Q2)**

We evaluate using chunk-level (entity-level) F1 with `seqeval`.


### **Task 6.1: Implement decoding for seqeval**
TODOs:
- compute argmax predictions
- map ids to tag strings

In [95]:
def decode_for_seqeval(logits, labels, attn, idx2tag):
    # TODO: compute argmax predictions
    pred_ids = logits.argmax(-1).detach().cpu().numpy()
    gold_ids = labels.detach().cpu().numpy()
    attn_np  = attn.detach().cpu().numpy()

    y_true, y_pred = [], []
    for i in range(pred_ids.shape[0]):
        t_true, t_pred = [], []
        for j in range(pred_ids.shape[1]):
            if attn_np[i, j] == 0:
                continue
            gid = int(gold_ids[i, j])
            pid = int(pred_ids[i, j])
            # TOD0: map ids to tag strings
            t_true.append(idx2tag[gid])
            t_pred.append(idx2tag[pid])
        y_true.append(t_true)
        y_pred.append(t_pred)
    return y_true, y_pred

In [96]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids)
        loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)

@torch.no_grad()
def evaluate_f1(model, loader, idx2tag, device, return_report=False):
    model.eval()
    all_true, all_pred = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attn = batch["attention_mask"].to(device)

        logits = model(input_ids)
        y_true, y_pred = decode_for_seqeval(logits, labels, attn, idx2tag)
        all_true.extend(y_true)
        all_pred.extend(y_pred)

    f1 = f1_score(all_true, all_pred)
    if return_report:
        report = classification_report(all_true, all_pred, digits=4)
        return f1, report
    return f1

In [97]:
# TODO 6.2: Train and report best validation F1, test F1 and classification report
epochs = 5
best_val = -1.0
best_state = None

for ep in range(1, epochs + 1):
    tr_loss =  train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_f1 = evaluate_f1(model, val_loader, idx2tag, device)
    print(f"Epoch {ep}: loss={tr_loss:.4f}, val_f1={val_f1:.4f}")

    if val_f1 > best_val:
        best_val = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print("Best validation F1:", best_val)

model.load_state_dict(best_state)
test_f1, test_report = evaluate_f1(model, test_loader, idx2tag, device, return_report=True)
print("Test F1:", test_f1)
print(test_report)

Epoch 1: loss=0.7571, val_f1=0.7736
Epoch 2: loss=0.3405, val_f1=0.8343
Epoch 3: loss=0.2407, val_f1=0.8567
Epoch 4: loss=0.1798, val_f1=0.8680
Epoch 5: loss=0.1375, val_f1=0.8740
Best validation F1: 0.87395344557917
Test F1: 0.877748056507565
              precision    recall  f1-score   support

        ADJP     0.5910    0.4817    0.5308       438
        ADVP     0.7618    0.6686    0.7122       866
       CONJP     0.6667    0.4444    0.5333         9
        INTJ     1.0000    0.5000    0.6667         2
         LST     0.0000    0.0000    0.0000         5
          NP     0.8668    0.8912    0.8788     12422
          PP     0.9512    0.9732    0.9621      4811
         PRT     0.6838    0.7547    0.7175       106
        SBAR     0.8755    0.8150    0.8441       535
          VP     0.8622    0.8452    0.8536      4658

   micro avg     0.8750    0.8805    0.8777     23852
   macro avg     0.7259    0.6374    0.6699     23852
weighted avg     0.8732    0.8805    0.8764     2385

## **Part 7 — Pre-trained GloVe Embeddings (Q3)**

We initialize the word embedding matrix using pre-trained GloVe vectors.

### **What is GloVe?**
**GloVe (Global Vectors for Word Representation)** is a set of **pre-trained word embeddings** that represent words as dense vectors.
These vectors are learned from large text corpora and capture semantic relationships.

In this homework, we use **Gensim** to load a standard GloVe model.


In [98]:
# Load GloVe vectors with gensim
import gensim.downloader as api

glove_name = "glove-wiki-gigaword-100"
glove_model = api.load(glove_name)
embedding_dim = glove_model.vector_size

print("Loaded glove:", glove_name, "dim:", embedding_dim, "vocab:", len(glove_model))
print("Example:", glove_model["computer"][:5])

Loaded glove: glove-wiki-gigaword-100 dim: 100 vocab: 400000
Example: [-0.16298   0.30141   0.57978   0.066548  0.45835 ]


In [99]:
# TODO 7.1: Build embedding matrix for our vocab
# For words not covered by pre-trained GloVe vectors, we initialize their
# embeddings randomly from a uniform distribution U(-0.1, 0.1)
embedding_matrix = np.random.uniform(
    -0.1, 0.1,
    (len(word2idx),embedding_dim)
).astype(np.float32)

oov = 0
for w, i in word2idx.items():
    if w == PAD_TOKEN:
        continue
    if w in glove_model:
        embedding_matrix[i] = glove_model[w]
    else:
        oov += 1

print("OOV words:", oov, "/", len(word2idx))

OOV words: 6148 / 18126


In [100]:
class BiLSTMTaggerPretrained(BiLSTMTagger):
    def __init__(self, vocab_size, num_tags, embedding_matrix, hidden_dim=128,
                 dropout=0.1, freeze_word_emb=False):
        super().__init__(vocab_size, num_tags, word_dim=embedding_matrix.shape[1],
                         hidden_dim=hidden_dim, dropout=dropout)
        with torch.no_grad():
            self.word_emb.weight.copy_(torch.tensor(embedding_matrix))
        self.word_emb.weight.requires_grad = (not freeze_word_emb)

In [101]:
# TODO 7.2: Train pretrained model and report best val F1 + test F1 + report
pretrained_model = BiLSTMTaggerPretrained(len(word2idx),len(tag2idx),embedding_matrix).to(device)  # TODO
optimizer_pretrained = torch.optim.AdamW(pretrained_model.parameters(),lr=2e-3)  # TODO

best_val_pre = -1.0
best_state_pre = None

for ep in range(1, epochs + 1):
    tr_loss = train_one_epoch(pretrained_model,train_loader,optimizer_pretrained,loss_fn,device)  # TODO
    val_f1 = evaluate_f1(pretrained_model,val_loader,idx2tag,device)  # TODO
    print(f"[GloVe] Epoch {ep}: loss={tr_loss:.4f}, val_f1={val_f1:.4f}")

    if val_f1 > best_val_pre:
        best_val_pre = val_f1
        best_state_pre = {k: v.detach().cpu().clone() for k, v in pretrained_model.state_dict().items()}

print("[GloVe] Best validation F1:", best_val_pre)

pretrained_model.load_state_dict(best_state_pre)
test_f1_pre, test_report_pre = evaluate_f1(pretrained_model, test_loader, idx2tag, device, return_report=True)
print("[GloVe] Test F1:", test_f1_pre)
print(test_report_pre)

[GloVe] Epoch 1: loss=0.6286, val_f1=0.8539
[GloVe] Epoch 2: loss=0.2219, val_f1=0.8933
[GloVe] Epoch 3: loss=0.1523, val_f1=0.9010
[GloVe] Epoch 4: loss=0.1139, val_f1=0.9064
[GloVe] Epoch 5: loss=0.0900, val_f1=0.9116
[GloVe] Best validation F1: 0.9115882972391374
[GloVe] Test F1: 0.9062487007857648
              precision    recall  f1-score   support

        ADJP     0.6533    0.6712    0.6622       438
        ADVP     0.7779    0.7644    0.7711       866
       CONJP     0.1667    0.2222    0.1905         9
        INTJ     0.0000    0.0000    0.0000         2
         LST     0.0000    0.0000    0.0000         5
          NP     0.8975    0.9178    0.9075     12422
          PP     0.9592    0.9784    0.9687      4811
         PRT     0.6720    0.7925    0.7273       106
        SBAR     0.8697    0.8860    0.8778       535
          VP     0.8955    0.8961    0.8958      4658

   micro avg     0.8987    0.9139    0.9062     23852
   macro avg     0.5892    0.6129    0.6001    

**Q4**

Report:
- a short interpretation (1–2 paragraphs) comparing random initialization and GloVe initialization in the report